W ramach tego noteboooka sprawdzimy, czy Twoja instalacja MNE-Python i najważniejszych bibliotek udała się.
Uruchom komórkę poniżej. *Nie powinna* pokazać Ci żadnego błędu

In [48]:
import mne
import numpy
import matplotlib
import scipy
mne.__version__     # sprawdźmy też od razu wersje, 1.11.0 lub wyższa

'1.13.2'

# Sprawdź interpreter notebooka

In [49]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Środowisko:", sys.prefix)

assert sys.version_info >= (3, 11), "Wybierz Python 3.11 lub nowszy."
assert sys.prefix != sys.base_prefix, "Wybierz kernel środowiska wirtualnego."
assert Path(sys.prefix).name == ".venv", "Wybierz .venv utworzone dla zajęć."   # jezeli uzywasz innej nazwy, zmień ją w tym miejscu
print("✓ Interpreter spełnia wymagania. Sprawdź też, czy ścieżka prowadzi do folderu zajęć.")


Python: 3.14.2
Interpreter: /Users/bejson/University/Dydaktyka/ADNeuroPy/.venv/bin/python
Środowisko: /Users/bejson/University/Dydaktyka/ADNeuroPy/.venv
✓ Interpreter spełnia wymagania. Sprawdź też, czy ścieżka prowadzi do folderu zajęć.


# Sprawdź biblioteki

In [50]:
import importlib
from importlib.metadata import version

packages = {
    "mne": "mne",
    "mne_qt_browser": "mne-qt-browser",
    "PyQt6.QtWidgets": "PyQt6",
    "numpy": "numpy",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "ipykernel": "ipykernel",
    "ipywidgets": "ipywidgets",
}
failed = []
for module, package in packages.items():
    try:
        importlib.import_module(module)
        print(f"✓ {package}: {version(package)}")
    except Exception as error:
        failed.append(package)
        print(f"✗ {package}: {type(error).__name__}: {error}")

assert not failed, f"Pakiety wymagające sprawdzenia: {', '.join(failed)}"
print("✓ Wszystkie wymagane pakiety są zainstalowane :)")


✓ mne: 1.13.2
✓ mne-qt-browser: 0.7.5
✓ PyQt6: 6.11.0
✓ numpy: 2.5.3
✓ scipy: 1.18.1
✓ matplotlib: 3.11.2
✓ pandas: 3.0.6
✓ ipykernel: 7.3.0
✓ ipywidgets: 8.1.9
✓ Wszystkie wymagane pakiety są zainstalowane :)


Możesz też wypisać wszystkie informacje dotyczące MNE. Jeżeli pojawi się błąd po stronie instalacji pakietów - szukaj tu brakujących elementów/złych wersji.

In [51]:
mne.sys_info()

Platform             macOS-26.2-arm64-arm-64bit-Mach-O
Python               3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]
Executable           /Users/bejson/University/Dydaktyka/ADNeuroPy/.venv/bin/python
CPU                  Apple M4 Pro (14 cores)
Memory               24.0 GiB

Core
├☑ mne               1.13.2 (unable to check for latest version on GitHub, SSL error)
├☑ numpy             2.5.3 (Accelerate, threads not introspectable)
├☑ scipy             1.18.1
└☑ matplotlib        3.11.2 (backend=inline)

Numerical (optional)
├☑ scikit-learn      1.9.1
├☑ threadpoolctl     3.7.0
├☑ numba             0.68.0
├☑ pandas            3.0.6
└☐ unavailable       nibabel, nilearn, dipy, openmeeg, python-picard, jamica, cupy, h5io, h5py

Visualization (optional)
├☑ qtpy              2.4.3 (PyQt6=6.11.0)
├☑ pyqtgraph         0.14.0
├☑ mne-qt-browser    0.7.5
├☑ ipywidgets        8.1.9
└☐ unavailable       pyvista, pyvistaqt, vtk, ipympl, trame, trame_cli

# Sprawdź czy na Twoim komputerze zadziała interaktywny wykres

In [52]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact

@interact(frequency=(1, 30, 1))
def show_signal(frequency=1):
    time = np.linspace(0, 1, 1000, endpoint=False)
    signal = np.sin(2 * np.pi * frequency * time)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(time, signal)
    ax.set(xlabel="Czas [s]", ylabel="Amplituda", title=f"Sin: {frequency} Hz", ylim=(-1.2, 1.2))
    ax.grid(alpha=0.3)
    plt.show()
    plt.close(fig)


interactive(children=(IntSlider(value=1, description='frequency', max=30, min=1), Output()), _dom_classes=('wi…

# Sprawdź interaktywną przeglądarkę
Otworzy się osobne okno obok VS Code (przy pierwszym uruchomieniu może to zająć chwilkę, nie panikuj przed 30 sekundami ;).
Zobaczysz trzy zasymulowane kanały EEG. Spróbuj przewijać sygnał i pobawić się przez chwilę tym oknem.
Potem zamknij interaktywne okno.

In [53]:
%gui qt6
import mne
import numpy as np

sfreq = 250.0
seconds = 10
time = np.arange(int(seconds * sfreq)) / sfreq
rng = np.random.default_rng(42)
data = np.array([20e-6 * np.sin(2 * np.pi * frequency * time)
    + rng.normal(0, 2e-6, time.size)
    for frequency in (5, 10, 15)])

info = mne.create_info(["EEG1", "EEG2", "EEG3"], sfreq, ch_types="eeg")
raw = mne.io.RawArray(data, info, verbose=False)
assert raw.get_data().shape == (3, 2500)
mne.viz.set_browser_backend("qt")
browser = raw.plot(duration=5, n_channels=3, scalings={"eeg": 50e-6}, block=False)

In [54]:
mne.sys_info()

Platform             macOS-26.2-arm64-arm-64bit-Mach-O
Python               3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 (clang-1600.0.26.6)]
Executable           /Users/bejson/University/Dydaktyka/ADNeuroPy/.venv/bin/python
CPU                  Apple M4 Pro (14 cores)
Memory               24.0 GiB

Core
├☑ mne               1.13.2 (unable to check for latest version on GitHub, SSL error)
├☑ numpy             2.5.3 (Accelerate, threads not introspectable)
├☑ scipy             1.18.1
└☑ matplotlib        3.11.2 (backend=inline)

Numerical (optional)
├☑ scikit-learn      1.9.1
├☑ threadpoolctl     3.7.0
├☑ numba             0.68.0
├☑ pandas            3.0.6
└☐ unavailable       nibabel, nilearn, dipy, openmeeg, python-picard, jamica, cupy, h5io, h5py

Visualization (optional)
├☑ qtpy              2.4.3 (PyQt6=6.11.0)
├☑ pyqtgraph         0.14.0
├☑ mne-qt-browser    0.7.5
├☑ ipywidgets        8.1.9
└☐ unavailable       pyvista, pyvistaqt, vtk, ipympl, trame, trame_cli